# Разметка дыхания эксперимента 2

**Статус:** активный производитель кандидатной и принятой вручную дыхательной
разметки. Автоматически найденные интервалы не являются проверенным входом
последующих расчётов.

Временные метки команд прибором не записывались. Даже принятые вручную
интервалы являются экспертной ретроспективной реконструкцией по памяти автора
и форме сигнала, а не непосредственно измеренными событиями. Для каждой
границы необходимо сохранять основание и степень уверенности.

Ноутбук выделен из исторического `08_Скетч_разметки.ipynb`. Исходная смешанная
реализация сохранена в
[`archive/legacy/11.90`](archive/legacy/11.90_Скетч_механических_точек.ipynb),
а происхождение ячеек — в
[`MIGRATION_MANIFEST_2026-08-26.json`](MIGRATION_MANIFEST_2026-08-26.json).


## Входы, допущения и выход

Входом служат внешние CSV эксперимента 2 с колонками времени и базового
импеданса боковой сборки. Пути, обезличенные идентификаторы, канонические
размеры каждого добровольца и исключённые дубликаты задаются во внешнем
локальном JSON по схеме
[`config/exp02_paths.example.json`](config/exp02_paths.example.json).

Алгоритм ищет два продолжительных участка с малой локальной вариабельностью.
Первый по времени участок получает кандидатную метку задержки после вдоха, а
второй — задержки после выдоха, поскольку такой порядок восстановлен и
подтверждён автором для эксперимента 2. Различие уровней `BASE_2` используется
только как диагностическое наблюдение и не меняет метки местами.

Поиск двух наиболее продолжительных спокойных участков является эвристикой.
Изменение контакта, базового уровня, движение или слабое дыхание могут создать
похожий фрагмент. Сопоставимость последовательно записанных размеров остаётся
допущением.

Каждый сопроводительный JSON получает полный SHA-256 исходной записи, версию
алгоритма, параметры эвристики и статус `pending_manual_review`. Последующие
расчёты принимают только `accepted_modes` из артефактов со статусом `accepted`.


In [ ]:
# Импорты и внешняя конфигурация
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CONFIG_ENV = "KALMYKOV_EXP02_CONFIG"
config_value = os.environ.get(CONFIG_ENV)
if not config_value:
    raise RuntimeError(
        f"Задайте {CONFIG_ENV}: путь к локальному JSON по схеме "
        "config/exp02_paths.example.json"
    )

CONFIG_PATH = Path(config_value).expanduser().resolve()
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"]).expanduser().resolve()
DERIVED_ROOT = Path(CONFIG["derived_root"]).expanduser().resolve()
subject_items = CONFIG["subjects"]
subject_ids = [item["subject_id"] for item in subject_items]
if len(subject_items) != 2 or len(subject_ids) != len(set(subject_ids)):
    raise ValueError("Эксперимент 2 должен содержать двух уникальных добровольцев")
SUBJECTS = {item["subject_id"]: item for item in subject_items}

OUT_DIR = DERIVED_ROOT / "exp02" / "annotations" / "breathing"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "TIME_s"
SEG_COL = "BASE_2_Ω"
ALGORITHM_VERSION = "exp02-breathing-heuristic-v2"
PARAMETERS = {
    "window_s": float(CONFIG["breathing_segmentation"]["window_s"]),
    "min_hold_s": float(CONFIG["breathing_segmentation"]["min_hold_s"]),
    "quiet_quantile": float(CONFIG["breathing_segmentation"]["quiet_quantile"]),
    "quiet_multiplier": float(CONFIG["breathing_segmentation"]["quiet_multiplier"]),
    "quiet_floor_ohm": float(CONFIG["breathing_segmentation"]["quiet_floor_ohm"]),
}
if not 0 < PARAMETERS["quiet_quantile"] < 100:
    raise ValueError("quiet_quantile задаётся в процентах от 0 до 100")
for name in ("window_s", "min_hold_s", "quiet_multiplier", "quiet_floor_ohm"):
    if not np.isfinite(PARAMETERS[name]) or PARAMETERS[name] <= 0:
        raise ValueError(f"{name} должен быть положительным")


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def resolve_under_data_root(path):
    resolved = path.expanduser().resolve()
    resolved.relative_to(DATA_ROOT)
    return resolved


def record_path(info, size_mm):
    directory = resolve_under_data_root(DATA_ROOT / info["data_subdir"])
    path = resolve_under_data_root(
        directory / f"{int(size_mm)}{info['filename_suffix']}.csv"
    )
    if not path.is_file():
        raise FileNotFoundError(f"Нет ожидаемой записи размера {size_mm} мм")
    return path


def read_record(path):
    frame = pd.read_csv(path, encoding="utf-8")
    missing = {TIME_COL, SEG_COL} - set(frame.columns)
    if missing:
        raise ValueError(f"В CSV отсутствуют обязательные столбцы: {sorted(missing)}")
    for column in (TIME_COL, SEG_COL):
        frame[column] = pd.to_numeric(frame[column], errors="raise")
        if not np.isfinite(frame[column].to_numpy(dtype=float)).all():
            raise ValueError(f"Столбец {column} содержит нечисловые значения")
    return frame


def sampling_frequency(frame):
    time = frame[TIME_COL].to_numpy(dtype=float)
    delta = np.diff(time)
    if len(delta) == 0 or np.any(~np.isfinite(delta)) or np.any(delta <= 0):
        raise ValueError("TIME_s должен быть конечным и строго возрастающим")
    median_dt = float(np.median(delta))
    jitter_fraction = float(np.median(np.abs(delta - median_dt)) / median_dt)
    return 1.0 / median_dt, jitter_fraction


def validate_subject_exclusions(subject_id, info):
    sizes = [int(value) for value in info["sizes_mm"]]
    if not sizes or len(sizes) != len(set(sizes)):
        raise ValueError(f"Для {subject_id} нужен уникальный непустой sizes_mm")
    exclusions = info.get("excluded_recordings", [])
    excluded_sizes = {int(item["size_mm"]) for item in exclusions}
    if excluded_sizes & set(sizes):
        raise ValueError(f"Исключённый размер включён в sizes_mm: {subject_id}")
    validated = []
    for item in exclusions:
        size_mm = int(item["size_mm"])
        duplicate_of = int(item["duplicate_of_size_mm"])
        excluded_path = record_path(info, size_mm)
        canonical_path = record_path(info, duplicate_of)
        excluded_sha = sha256_file(excluded_path)
        canonical_sha = sha256_file(canonical_path)
        if excluded_sha != canonical_sha:
            raise ValueError(
                f"Заявленный дубликат {subject_id}, {size_mm} мм "
                f"не совпадает с {duplicate_of} мм"
            )
        validated.append({
            "subject_id": subject_id,
            "size_mm": size_mm,
            "duplicate_of_size_mm": duplicate_of,
            "sha256": excluded_sha,
            "reason": item.get("reason", "byte_identical_duplicate"),
        })
    return validated


def iter_recordings():
    for subject_id, info in SUBJECTS.items():
        for size_mm in info["sizes_mm"]:
            yield subject_id, info, int(size_mm), record_path(info, size_mm)


In [ ]:
# Эвристическое выделение кандидатов дыхательных режимов
def segment_breathing(frame, parameters=PARAMETERS):
    time = frame[TIME_COL].to_numpy(dtype=float)
    signal = frame[SEG_COL].to_numpy(dtype=float)
    fs, _ = sampling_frequency(frame)

    window = max(5, int(round(parameters["window_s"] * fs)))
    rolling_std = (
        pd.Series(signal)
        .rolling(window, center=True, min_periods=max(3, window // 2))
        .std()
        .to_numpy()
    )
    finite_std = rolling_std[np.isfinite(rolling_std)]
    if len(finite_std) == 0:
        return None
    threshold = max(
        parameters["quiet_floor_ohm"],
        float(
            np.percentile(finite_std, parameters["quiet_quantile"])
            * parameters["quiet_multiplier"]
        ),
    )
    quiet = np.isfinite(rolling_std) & (rolling_std < threshold)

    runs = []
    index = 0
    while index < len(quiet):
        if not quiet[index]:
            index += 1
            continue
        stop = index
        while stop < len(quiet) and quiet[stop]:
            stop += 1
        if stop > index and time[stop - 1] - time[index] >= parameters["min_hold_s"]:
            runs.append((index, stop))
        index = stop

    runs.sort(key=lambda item: -(time[item[1] - 1] - time[item[0]]))
    holds = sorted(runs[:2], key=lambda item: item[0])
    if len(holds) < 2:
        return None

    inhale, exhale = holds
    inhale_level = float(np.median(signal[inhale[0]:inhale[1]]))
    exhale_level = float(np.median(signal[exhale[0]:exhale[1]]))

    def closed_interval(segment):
        return [float(time[segment[0]]), float(time[segment[1] - 1])]

    deep_start_index = min(inhale[1], len(time) - 1)
    modes = {
        "спокойное": [float(time[0]), float(time[inhale[0]])],
        "задержка_вдох": closed_interval(inhale),
        "глубокое": [float(time[deep_start_index]), float(time[exhale[0]])],
        "задержка_выдох": closed_interval(exhale),
    }
    return {
        "candidate_modes": modes,
        "hold_levels_ohm": {
            "вдох": inhale_level,
            "выдох": exhale_level,
        },
        "observed_level_relation": {
            "inhale_higher_than_exhale": bool(inhale_level > exhale_level),
            "used_for_mode_assignment": False,
        },
        "heuristic": {
            **parameters,
            "quiet_threshold_ohm": threshold,
        },
    }


In [ ]:
# Проверка исключений и построение отдельных дыхательных sidecar-файлов
validated_exclusions = []
for subject_id, info in SUBJECTS.items():
    validated_exclusions.extend(validate_subject_exclusions(subject_id, info))

for item in validated_exclusions:
    print(
        "Подтверждён байт-в-байт дубликат:",
        item["subject_id"],
        f"{item['size_mm']}→{item['duplicate_of_size_mm']} мм",
    )

records = []
record_ids = set()
for subject_id, info, size_mm, source_path in iter_recordings():
    frame = read_record(source_path)
    fs_hz, jitter_fraction = sampling_frequency(frame)
    candidate = segment_breathing(frame)
    if candidate is None:
        print("Не найдены две кандидатные задержки:", subject_id, size_mm)
        continue

    input_sha256 = sha256_file(source_path)
    record_id = input_sha256[:16]
    if record_id in record_ids:
        raise ValueError("Коллизия record_id среди независимых записей")
    record_ids.add(record_id)
    relative_path = source_path.relative_to(DATA_ROOT).as_posix()
    output = {
        "schema_version": 2,
        "annotation_type": "breathing",
        "algorithm_version": ALGORITHM_VERSION,
        "record_id": record_id,
        "subject_id": subject_id,
        "size_mm": size_mm,
        "input": {
            "relative_path": relative_path,
            "sha256": input_sha256,
            "sampling_frequency_hz": fs_hz,
            "sampling_frequency_source": "median_diff_TIME_s",
            "relative_time_step_jitter": jitter_fraction,
        },
        **candidate,
        "accepted_modes": None,
        "assumptions": [
            "two_longest_quiet_segments_are_breath_holds",
            "first_hold_is_inhale_and_second_hold_is_exhale_by_author_reconstructed_order",
            "recordings_of_different_sizes_are_comparable",
        ],
        "qc": {
            "status": "pending_manual_review",
            "reviewer": None,
            "reviewed_at": None,
            "notes": None,
            "history": [],
        },
    }
    output_path = OUT_DIR / f"{record_id}.json"
    if output_path.exists():
        existing = json.loads(output_path.read_text(encoding="utf-8"))
        if existing.get("input", {}).get("sha256") != input_sha256:
            raise RuntimeError(f"Конфликт входного SHA-256 для {record_id}")
        if existing.get("subject_id") != subject_id or existing.get("size_mm") != size_mm:
            raise RuntimeError(f"Конфликт происхождения sidecar {record_id}")
        if existing.get("algorithm_version") != ALGORITHM_VERSION:
            raise RuntimeError(
                f"Sidecar {record_id} создан другой версией алгоритма; "
                "его нужно отдельно пересмотреть или архивировать"
            )
        if (
            existing.get("qc", {}).get("status") == "accepted"
            and not existing.get("accepted_modes")
        ):
            raise RuntimeError(
                f"Принятый sidecar {record_id} не содержит accepted_modes"
            )
        records.append(existing)
        print("Сохранён существующий sidecar:", record_id, existing["qc"]["status"])
        continue

    output_path.write_text(
        json.dumps(output, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    records.append(output)
    print("Создан кандидат:", record_id, size_mm, output["qc"]["status"])

expected_count = int(CONFIG["expected_independent_record_count"])
configured_count = sum(len(item["sizes_mm"]) for item in subject_items)
if configured_count != expected_count:
    raise ValueError(
        f"В sizes_mm задано {configured_count} записей вместо {expected_count}"
    )
if len(records) != expected_count:
    raise RuntimeError(
        f"Получено {len(records)} sidecar-файлов вместо {expected_count}; "
        "проверьте ненайденные задержки"
    )
print("Кандидатных или ранее сохранённых sidecar-файлов:", len(records))


## Ручной контроль качества и принятие разметки

Для каждого файла необходимо проверить форму `BASE_2_Ω`, согласованность
кандидатных интервалов с восстановленным по памяти порядком режимов и
отсутствие контактных или двигательных артефактов. Решение принимается в этом
же ноутбуке через `REVIEW_DECISION`. В примечании к решению необходимо указать,
какие особенности сигнала использованы для каждой границы и какие участки
остаются неоднозначными.

Статус `accepted` требует имени проверяющего и явно заданных
`accepted_modes`; автоматические кандидаты не принимаются молча. Решение
`rejected` сохраняет причину отказа. Повторное построение кандидатов не
перезаписывает существующий сопроводительный файл и, следовательно, не стирает
ручной контроль.


In [ ]:
# Ручной просмотр и явное принятие или отклонение разметки
CHECK_RECORD_ID = None
REVIEW_DECISION = None
# Пример решения:
# REVIEW_DECISION = {
#     "record_id": "<record_id>",
#     "status": "accepted",  # accepted или rejected
#     "reviewer": "<reviewer>",
#     "accepted_modes": {
#         "спокойное": [0.0, 10.0],
#         "задержка_вдох": [10.0, 20.0],
#         "глубокое": [20.0, 40.0],
#         "задержка_выдох": [40.0, 50.0],
#     },
#     "notes": "<основание решения>",
# }

MODE_ORDER = [
    "спокойное",
    "задержка_вдох",
    "глубокое",
    "задержка_выдох",
]


def validate_modes(modes, start_s, stop_s):
    if not isinstance(modes, dict) or list(modes) != MODE_ORDER:
        raise ValueError(f"Режимы должны идти в порядке {MODE_ORDER}")
    previous_stop = start_s
    normalized = {}
    for name in MODE_ORDER:
        interval = modes[name]
        if not isinstance(interval, list) or len(interval) != 2:
            raise ValueError(f"Интервал {name} должен содержать начало и конец")
        left, right = map(float, interval)
        if not np.isfinite([left, right]).all() or not start_s <= left < right <= stop_s:
            raise ValueError(f"Недопустимые границы интервала {name}")
        if left < previous_stop:
            raise ValueError(f"Интервалы перекрываются перед {name}")
        normalized[name] = [left, right]
        previous_stop = right
    return normalized


def apply_review(decision):
    record_id = decision["record_id"]
    sidecar_path = OUT_DIR / f"{record_id}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_under_data_root(
        DATA_ROOT / annotation["input"]["relative_path"]
    )
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")

    status = decision["status"]
    reviewer = str(decision.get("reviewer", "")).strip()
    if status not in {"accepted", "rejected"} or not reviewer:
        raise ValueError("Нужны статус accepted/rejected и имя проверяющего")

    accepted_modes = None
    if status == "accepted":
        frame = read_record(source_path)
        time = frame[TIME_COL].to_numpy(dtype=float)
        accepted_modes = validate_modes(
            decision.get("accepted_modes"),
            float(time[0]),
            float(time[-1]),
        )

    reviewed_at = datetime.now(timezone.utc).isoformat()
    previous_qc = annotation.get("qc", {})
    history = list(previous_qc.get("history", []))
    history.append({
        "status": previous_qc.get("status"),
        "reviewer": previous_qc.get("reviewer"),
        "reviewed_at": previous_qc.get("reviewed_at"),
        "notes": previous_qc.get("notes"),
    })
    annotation["accepted_modes"] = accepted_modes
    annotation["qc"] = {
        "status": status,
        "reviewer": reviewer,
        "reviewed_at": reviewed_at,
        "notes": decision.get("notes"),
        "history": history,
    }
    sidecar_path.write_text(
        json.dumps(annotation, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    return annotation


if REVIEW_DECISION is not None:
    reviewed = apply_review(REVIEW_DECISION)
    print(reviewed["record_id"], reviewed["qc"]["status"])

if CHECK_RECORD_ID is None:
    print("Задайте CHECK_RECORD_ID для просмотра конкретной записи.")
else:
    sidecar_path = OUT_DIR / f"{CHECK_RECORD_ID}.json"
    annotation = json.loads(sidecar_path.read_text(encoding="utf-8"))
    source_path = resolve_under_data_root(
        DATA_ROOT / annotation["input"]["relative_path"]
    )
    if sha256_file(source_path) != annotation["input"]["sha256"]:
        raise RuntimeError("SHA-256 исходного CSV изменился после разметки")

    frame = read_record(source_path)
    time = frame[TIME_COL].to_numpy(dtype=float)
    modes = (
        annotation.get("accepted_modes")
        if annotation.get("qc", {}).get("status") == "accepted"
        else annotation["candidate_modes"]
    )
    fig, axis = plt.subplots(figsize=(14, 4))
    axis.plot(time, frame[SEG_COL], color="navy", linewidth=0.7)
    for name, (start, stop) in modes.items():
        axis.axvspan(start, stop, alpha=0.2, label=name)
    axis.set_xlabel("Время, с")
    axis.set_ylabel("BASE_2, Ом")
    axis.set_title(
        f"{CHECK_RECORD_ID}: дыхательная разметка; "
        f"QC={annotation['qc']['status']}"
    )
    axis.legend()
    plt.tight_layout()
    plt.show()


## Выход и зависимые этапы

Канонические сопроводительные файлы находятся во внешнем каталоге
`derived/exp02/annotations/breathing/`. В Git допускаются только схема и
обезличенный пример.

[`11.02`](11.02_Разметка_ЭКГ_эксперимента_2.ipynb) связывает ЭКГ-разметку с
той же записью по `record_id` и полному SHA-256 исходного CSV. [`10.01`](10.01_QC_эксперимента_2_порядок_и_каналы.ipynb) и
последующие расчёты должны отклонять дыхательные артефакты без статуса
`accepted`.
